In [1]:
import os
import sys
import argparse
import torch
import geopandas as gpd
import numpy as np
import shutil
import json
from pathlib import Path
from model import get_model
from config import Config
from utils import setup_logging
from data_loader import EarthquakeTileDataset, custom_collate
from transforms import get_val_test_transforms
from tqdm import tqdm
from argparse import Namespace

# Simular argumentos como si vinieran de la línea de comandos
args = Namespace(
    output_dir='output/',
    config='config.yaml',
    device='cuda:0',
    threshold=0.5,
    split='test',
    batch_size=8,
    num_workers=2,
    model_path='checkpoint.pth'
)


c:\Users\esteb\.conda\envs\earthquake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Argumentos ---
def parse_args():
    parser = argparse.ArgumentParser(description="Generar predicciones para submission de la competición.")
    parser.add_argument('--model_path', type=str, default='outputs/models/checkpoint_epoch_5.pth', help='Ruta al modelo .pth')
    parser.add_argument('--input_dir', type=str, default='submissions/submission_1', help='Carpeta con archivos .gpkg a predecir')
    parser.add_argument('--output_dir', type=str, default='submissions/submission_3', help='Carpeta donde guardar la submission')
    parser.add_argument('--config', type=str, default='config.yaml', help='Archivo de configuración')
    parser.add_argument('--device', type=str, default='cuda', help='cpu o cuda')
    parser.add_argument('--split', type=str, default='test', help='Split de preprocesado a usar (test o full)')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size para inferencia')
    parser.add_argument('--threshold', type=float, default=None, help='Umbral para binarización (override config)')
    parser.add_argument('--num_workers', type=int, default=8, help='Número de workers para DataLoader')
    return parser.parse_args()

def load_checkpoint_to_model(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
    return model


In [ ]:
#moverse de la carpeta src al principal
os.chdir(os.path.dirname(os.path.abspath(os.getcwd())))

In [10]:
args = Namespace(
    model_path='outputs/models/checkpoint_epoch_2.pth',
    input_dir='submissions/submission_1',
    output_dir='submissions/submission_3',
    config='config.yaml',
    device='cuda:0',
    split='test',
    batch_size=8,
    threshold=0.5,
    num_workers=2
)

os.makedirs(args.output_dir, exist_ok=True)
# Cargar configuración y modelo
config = Config.from_yaml(args.config)
config.hardware.device = args.device
device = torch.device(args.device)
# Override threshold si se pasa por argumento
threshold = args.threshold if args.threshold is not None else getattr(config.training, 'threshold', 0.5)
max_channels = getattr(config.data, 'max_channels', 3)
# Dataset de test
data_root = getattr(config.data, 'data_root', 'preprocessed')

# DataLoader para test
test_loader_path = Path('outputs/dataloaders/dataloader_test_full_post_only.pt')
dataset = torch.load(test_loader_path,weights_only=False)  # debe ser el Dataset, no DataLoader
test_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_workers,
    pin_memory=True,
    collate_fn=custom_collate
)

# Cargar modelo
model = get_model(config.model)
model = load_checkpoint_to_model(model, args.model_path, device)
model.eval()
model.to(device)

# --- Inferencia por batches ---
pred_dict = {}
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inferencia'): 
        images = batch['image'].to(device)  # [B, C, H, W]
        logits = model(images)
        if logits.ndim == 1:
            logits = logits.unsqueeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()  # [B, 1] o [B]
        # Si probs es [B,1], aplanar
        if probs.ndim == 2 and probs.shape[1] == 1:
            probs = probs[:, 0]
        preds = (probs > threshold).astype(int)
        fids = batch['fid']
        events = batch['event']
        for i in range(len(preds)):
            pred_dict[(events[i], fids[i])] = int(preds[i])

2025-07-20 11:16:08.242 | INFO     | model:get_model:523 - Modelo creado: simple_classifier
2025-07-20 11:16:08.243 | INFO     | model:get_model:524 - Parámetros totales: 25,232,129
2025-07-20 11:16:08.244 | INFO     | model:get_model:525 - Parámetros entrenables: 25,232,129
Inferencia:   0%|          | 0/767 [00:04<?, ?it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\esteb\.conda\envs\earthquake\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "c:\Users\esteb\.conda\envs\earthquake\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
TypeError: 'DataLoader' object is not subscriptable


In [ ]:
# --- Procesar cada archivo .gpkg ---
gpkg_files = [f for f in os.listdir(args.input_dir) if f.endswith('.gpkg')]
for fname in tqdm(gpkg_files, desc='Procesando archivos .gpkg'):
    in_path = os.path.join(args.input_dir, fname)
    gdf = gpd.read_file(in_path)
    # Extraer nombre base de ciudad/evento del nombre del archivo
    base = os.path.splitext(fname)[0]
    # Tomar la parte de ciudad/evento (antes de '_testing')
    event_name = base.split('_')[0]
    # Buscar columna fid_<base> (ignorando mayúsculas/minúsculas)
    fid_col = None
    for col in gdf.columns:
        if col.lower().startswith('fid_') and base.lower() in col.lower():
            fid_col = col
            break
    if fid_col is None:
        for col in gdf.columns:
            if col.lower().startswith('fid_'):
                fid_col = col
                break
    if fid_col is None and 'fid' in gdf.columns:
        fid_col = 'fid'
    if fid_col is None:
        raise ValueError(f"No se encontró columna fid_<basename>, ni ninguna columna que empiece por fid_ ni 'fid' en {fname}")
    damages = []
    for idx, row in gdf.iterrows():
        fid = row[fid_col]
        pred = pred_dict.get((event_name, fid), 0)
        damages.append(pred)
    # DEBUG: Mostrar ejemplos de predicciones y conteo de daños
    print(f"[DEBUG] {fname} - Ejemplo de predicciones:", damages[:20])
    print(f"[DEBUG] {fname} - Conteo de daños:", sum(damages))
    gdf['damage'] = damages
    out_path = os.path.join(args.output_dir, fname)
    gdf.to_file(out_path, driver='GPKG')
# Comprimir la carpeta de submission
zip_base = os.path.abspath(args.output_dir)
shutil.make_archive(zip_base, 'zip', args.output_dir)
print(f"Submission generada en: {args.output_dir}, {zip_base}.zip")